In [14]:
import sys, os, math, random, shutil, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image


In [25]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT = Path('/juanj/escritorio/SemaforoIA/')
IMAGENES_DIR = DRIVE_ROOT / 'imagenes'
XML_DIR = DRIVE_ROOT / 'annotations_xml'
WORK_DIR = Path('/content/semaforo')
ANN_DIR = WORK_DIR / 'annotations'
TRAIN_DIR = WORK_DIR / 'training'
EXPORT_DIR = WORK_DIR / 'exported_model'
OUT_DIR = WORK_DIR / 'outputs'
for p in [ANN_DIR, TRAIN_DIR, EXPORT_DIR, OUT_DIR]: p.mkdir(parents=True, exist_ok=True)

CLASS_NAME = 'carro'
NUM_CLASSES = 1
BATCH_SIZE = 4
EPOCHS = 30
TRAIN_FRACTION = 0.8
SCORE_THRESHOLD = 0.30
IOU_THRESHOLD = 0.50
EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
image_paths = sorted(p for p in IMAGENES_DIR.glob('*') if p.suffix.lower() in EXTS)
missing = [p.name for p in image_paths if not (XML_DIR / f'{p.stem}.xml').exists()]
if missing: raise ValueError(f'Faltan XML para estas imágenes (primeras 10): {missing[:10]}')
print('Imágenes totales:', len(image_paths))

Imágenes totales: 0


In [ ]:
def iou(a,b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1]); x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]); ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(aa+ab-inter+1e-9)

def ground_truth(path):
    w,h,objs=read_voc(XML_DIR/f'{path.stem}.xml')
    return [[x1,y1,x2,y2] for name,x1,y1,x2,y2 in objs if name==CLASS_NAME]

def predictions(path):
    img=load_np(path); h,w=img.shape[:2]; d=detect(img); boxes=[]; scores=[]
    for box,score,cls in zip(d['detection_boxes'][0],d['detection_scores'][0],d['detection_classes'][0]):
        if int(cls)!=1 or float(score)<SCORE_THRESHOLD: continue
        ymin,xmin,ymax,xmax=box
        boxes.append([xmin*w,ymin*h,xmax*w,ymax*h]); scores.append(float(score))
    return boxes,scores

def match(gt,pred):
    candidates=sorted([(iou(g,p),i,j) for i,g in enumerate(gt) for j,p in enumerate(pred)], reverse=True)
    usedg=set(); usedp=set()
    for score,i,j in candidates:
        if score<IOU_THRESHOLD: break
        if i not in usedg and j not in usedp: usedg.add(i); usedp.add(j)
    tp=len(usedg); return tp, len(pred)-tp, len(gt)-tp

rows=[]
for p in test_paths:
    gt=ground_truth(p); pred,scores=predictions(p); tp,fp,fn=match(gt,pred)
    rows.append({'imagen':p.name,'carros_reales':len(gt),'carros_detectados':len(pred),
                 'confianza_media':np.mean(scores) if scores else 0.0,'TP':tp,'FP':fp,'FN':fn})
results=pd.DataFrame(rows)
results.to_csv(OUT_DIR/'resultados_experimento.csv',index=False)
TP,FP,FN=results.TP.sum(),results.FP.sum(),results.FN.sum()
precision=TP/(TP+FP+1e-9); recall=TP/(TP+FN+1e-9); f1=2*precision*recall/(precision+recall+1e-9)
print(f'TP={TP} FP={FP} FN={FN} | precisión={precision:.3f} recall={recall:.3f} F1={f1:.3f}')
display(results.head())

In [ ]:
# Matriz adaptada a detección: TP, FN y FP. No es una matriz multiclase estándar.
cm = np.array([[TP, FN],[FP, 0]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Emparejado','No detectado'],
            yticklabels=['Carro real','Falsa detección'])
plt.title(f'Matriz resumida de detección (IoU >= {IOU_THRESHOLD})')
plt.tight_layout(); plt.savefig(OUT_DIR/'matriz_confusion.png',dpi=300); plt.show()

In [ ]:
cols=['carros_reales','carros_detectados','confianza_media','TP','FP','FN']
plt.figure(figsize=(9,7))
sns.heatmap(results[cols].corr(numeric_only=True),annot=True,cmap='coolwarm',vmin=-1,vmax=1,fmt='.2f')
plt.title('Matriz de correlación de resultados'); plt.tight_layout()
plt.savefig(OUT_DIR/'matriz_correlacion.png',dpi=300); plt.show()

plt.figure(figsize=(11,5))
plt.plot(results.carros_reales.to_numpy(),label='Reales',marker='o')
plt.plot(results.carros_detectados.to_numpy(),label='Detectados',marker='x')
plt.xlabel('Imagen de prueba'); plt.ylabel('Número de carros'); plt.title('Conteo real vs. detectado')
plt.legend(); plt.tight_layout(); plt.savefig(OUT_DIR/'conteos.png',dpi=300); plt.show()

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def lime_predict(batch):
    vals=[]
    for image in batch:
        d=detect(image.astype(np.uint8))
        scores=d['detection_scores'][0]; classes=d['detection_classes'][0].astype(int)
        car=scores[(classes==1)&(scores>=0.01)]
        score=float(car.max()) if len(car) else 0.0
        vals.append([1-score,score])
    return np.asarray(vals)

lime_explainer=lime_image.LimeImageExplainer()
lime_img=load_np(test_paths[0])
explanation=lime_explainer.explain_instance(lime_img,lime_predict,top_labels=2,hide_color=0,num_samples=300)
lime_vis,lime_mask=explanation.get_image_and_mask(1,positive_only=True,num_features=8,hide_rest=False)
plt.figure(figsize=(10,7)); plt.imshow(mark_boundaries(lime_vis/255.0 if lime_vis.max()>1 else lime_vis,lime_mask))
plt.title('LIME: regiones que influyen en la puntuación del detector'); plt.axis('off')
plt.tigh

In [ ]:
print('Firmas SavedModel:', list(detect_fn.signatures.keys()))
sig=detect_fn.signatures.get('serving_default')
if sig:
    print('Entradas:',sig.structured_input_signature)
    print('Salidas:',list(sig.structured_outputs.keys()))
print('Este diagnóstico no genera un mapa Grad-CAM. Para implementarlo con rigor,')
print('hay que identificar una capa convolucional interna y definir la puntuación de la detección objetivo.')

In [ ]:
def count_lanes(path, roi_left=(0,0,0.5,1), roi_right=(0.5,0,1,1)):
    img=load_np(path); d=detect(img)
    counts={'izquierdo':0,'derecho':0}
    for box,score,cls in zip(d['detection_boxes'][0],d['detection_scores'][0],d['detection_classes'][0]):
        if float(score)<SCORE_THRESHOLD or int(cls)!=1: continue
        ymin,xmin,ymax,xmax=box; cx=(xmin+xmax)/2; cy=(ymin+ymax)/2
        if roi_left[0] <= cx <= roi_left[2] and roi_left[1] <= cy <= roi_left[3]:
            counts['izquierdo']+=1
        elif roi_right[0] <= cx <= roi_right[2] and roi_right[1] <= cy <= roi_right[3]:
            counts['derecho']+=1
    if counts['izquierdo']>counts['derecho']: decision='Priorizar carril izquierdo'
    elif counts['derecho']>counts['izquierdo']: decision='Priorizar carril derecho'
    else: decision='Empate: mantener ciclo normal'
    return counts,decision

conteo,decision=count_lanes(test_paths[0])
print('Conteo:',conteo)
print('Decisión experimental:',decision)
print('Ajusta ROI_LEFT/ROI_RIGHT a la posición real de la cámara antes de usarlo.')